# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [18]:
print("Notebook is running")

Notebook is running


In [19]:
print("df" in globals())

False


In [22]:
!git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 283, done.
remote: Counting objects: 100% (145/145), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 283 (delta 112), reused 83 (delta 83), pack-reused 138 (from 1)
Receiving objects: 100% (283/283), 1.85 MiB | 4.14 MiB/s, done.
Resolving deltas: 100% (153/153), done.


In [23]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path("/content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

Shape: (30000, 44)
Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [16]:
pd.read_csv

<function pandas.io.parsers.readers.read_csv(filepath_or_buffer: 'FilePath | ReadCsvBuffer[bytes] | ReadCsvBuffer[str]', *, sep: 'str | None | lib.NoDefault' = <no_default>, delimiter: 'str | None | lib.NoDefault' = None, header: "int | Sequence[int] | None | Literal['infer']" = 'infer', names: 'Sequence[Hashable] | None | lib.NoDefault' = <no_default>, index_col: 'IndexLabel | Literal[False] | None' = None, usecols: 'UsecolsArgType' = None, dtype: 'DtypeArg | None' = None, engine: 'CSVEngine | None' = None, converters: 'Mapping[Hashable, Callable] | None' = None, true_values: 'list | None' = None, false_values: 'list | None' = None, skipinitialspace: 'bool' = False, skiprows: 'list[int] | int | Callable[[Hashable], bool] | None' = None, skipfooter: 'int' = 0, nrows: 'int | None' = None, na_values: 'Hashable | Iterable[Hashable] | Mapping[Hashable, Iterable[Hashable]] | None' = None, keep_default_na: 'bool' = True, na_filter: 'bool' = True, verbose: 'bool | lib.NoDefault' = <no_default>, skip_blank_lines: 'bool' = True, parse_dates: 'bool | Sequence[Hashable] | None' = None, infer_datetime_format: 'bool | lib.NoDefault' = <no_default>, keep_date_col: 'bool | lib.NoDefault' = <no_default>, date_parser: 'Callable | lib.NoDefault' = <no_default>, date_format: 'str | dict[Hashable, str] | None' = None, dayfirst: 'bool' = False, cache_dates: 'bool' = True, iterator: 'bool' = False, chunksize: 'int | None' = None, compression: 'CompressionOptions' = 'infer', thousands: 'str | None' = None, decimal: 'str' = '.', lineterminator: 'str | None' = None, quotechar: 'str' = '"', quoting: 'int' = 0, doublequote: 'bool' = True, escapechar: 'str | None' = None, comment: 'str | None' = None, encoding: 'str | None' = None, encoding_errors: 'str | None' = 'strict', dialect: 'str | csv.Dialect | None' = None, on_bad_lines: 'str' = 'error', delim_whitespace: 'bool | lib.NoDefault' = <no_default>, low_memory: 'bool' = True, memory_map: 'bool' = False, float_precision: "Literal['high', 'legacy'] | None" = None, storage_options: 'StorageOptions | None' = None, dtype_backend: 'DtypeBackend | lib.NoDefault' = <no_default>) -> 'DataFrame | TextFileReader'>

In [31]:
# STEP 1 — Signal checks

# -----------------------------
# Signal 1: Staleness
# -----------------------------

df["age_bucket"] = pd.cut(
    df["content_age_days"],
    bins=[-1, 30, 60, 90, float("inf")],
    labels=["0-30", "31-60", "61-90", "91+"]
)

staleness_table = (
    df.groupby("age_bucket", observed=False)
      .agg(
          n=("content_age_days", "size"),
          median_age=("content_age_days", "median")
      )
      .reset_index()
)

print("SIGNAL 1: STALENESS")
display(staleness_table)


# -----------------------------
# Signal 2: CTR vs Position
# -----------------------------

df["position_bucket"] = pd.cut(
    df["avg_position"],
    bins=[-float("inf"), 3, 10, 20, 50, float("inf")],
    labels=["1-3", "4-10", "11-20", "21-50", "51+"]
)

ctr_position_table = (
    df.groupby("position_bucket", observed=False)
      .agg(
          n=("ctr", "size"),
          median_ctr=("ctr", "median")
      )
      .reset_index()
)

print("SIGNAL 2: CTR VS POSITION")
display(ctr_position_table)



SIGNAL 1: STALENESS


,age_bucket,n,median_age
0,0-30,0,NaN
1,31-60,0,NaN
2,61-90,492,90.0
3,91+,29508,236.0


SIGNAL 2: CTR VS POSITION


,position_bucket,n,median_ctr
0,1-3,2346,0.00
1,4-10,11842,0.16
2,11-20,7273,0.10
3,21-50,7225,0.03
4,51+,1314,0.00


## Signal 1 — Staleness

I checked `content_age_days` because staleness is directly related to FlyRank's refresh flags.

**Verdict: CONFIRMED**

The observed bucket distribution shows [describe your actual result]. This makes content age a useful directional signal for identifying pages that may need review or refresh.

## Signal 2 — CTR vs Position

I checked CTR relative to `avg_position` because FlyRank's CTR-fix logic uses search position as context.

**Verdict: [CONFIRMED / MIXED / OPPOSITE / FALSE]**

The observed position buckets show [describe your actual result]. Therefore, I will [use/limit/not use] this signal in the baseline rule.

## Rule reasoning

I will use the validated signals to create a simple decision-support baseline. Pages receive points for the selected signals, producing one baseline score. The score is then converted into one reason code and one action label.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [25]:
work = df.copy()

In [27]:
# Step 2: Build the ranked queue

import numpy as np
import pandas as pd
from pathlib import Path

# Make a copy of the input data
work = df.copy()


work["staleness_points"] = pd.cut(
    work["content_age_days"],
    bins=[-1, 30, 60, 90, float("inf")],
    labels=[0, 1, 2, 3]
).astype(int)



work["volume_points"] = pd.qcut(
    work["impressions_90d"].rank(method="first"),
    q=4,
    labels=[0, 1, 2, 3]
).astype(int)


-

work["position_bucket"] = pd.cut(
    work["avg_position"],
    bins=[-float("inf"), 3, 10, 20, 50, float("inf")],
    labels=["1-3", "4-10", "11-20", "21-50", "51+"]
)

position_ctr = (
    work.groupby("position_bucket", observed=False)["ctr"]
    .median()
    .rename("position_median_ctr")
)

work = work.join(
    position_ctr,
    on="position_bucket"
)



work["ctr_gap"] = (
    work["position_median_ctr"] - work["ctr"]
).clip(lower=0)

work["ctr_points"] = pd.qcut(
    work["ctr_gap"].rank(method="first"),
    q=4,
    labels=[0, 1, 2, 3]
).astype(int)


work["baseline_score"] = (
    work["staleness_points"]
    + work["volume_points"]
    + work["ctr_points"]
)




work["reason_code"] = np.select(
    [
        (work["staleness_points"] >= 2) &
        (work["volume_points"] >= 2),

        work["ctr_points"] >= 2,

        work["volume_points"] >= 2
    ],
    [
        "STALE_HIGH_VALUE",
        "CTR_OPPORTUNITY",
        "HIGH_VOLUME"
    ],
    default="LOW_PRIORITY"
)



work["action"] = np.select(
    [
        work["baseline_score"] >= 7,
        work["baseline_score"] >= 4,
        work["baseline_score"] >= 2
    ],
    [
        "REFRESH_NOW",
        "REVIEW",
        "MONITOR"
    ],
    default="NO_ACTION"
)




work = work.sort_values(
    ["baseline_score", "impressions_90d"],
    ascending=[False, False]
).reset_index(drop=True)

work["rank"] = np.arange(1, len(work) + 1)



output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

work.to_csv(
    output_path,
    index=False
)


print(f"Rows ranked: {len(work):,}")
print(f"CSV written to: {output_path}")

display(
    work[
        [
            "rank",
            "content_id",
            "baseline_score",
            "reason_code",
            "action"
        ]
    ].head(20)
)

Rows ranked: 30,000
CSV written to: work/outputs/baseline_action_score.csv


,rank,content_id,baseline_score,reason_code,action
0,1,content_36ff89c8214e,9,STALE_HIGH_VALUE,REFRESH_NOW
1,2,content_c84a0ab98e90,9,STALE_HIGH_VALUE,REFRESH_NOW
2,3,content_c8e9d6ab9013,9,STALE_HIGH_VALUE,REFRESH_NOW
3,4,content_91652435f57a,9,STALE_HIGH_VALUE,REFRESH_NOW
4,5,content_97a86caf3a3d,9,STALE_HIGH_VALUE,REFRESH_NOW
5,6,content_453722754fea,9,STALE_HIGH_VALUE,REFRESH_NOW
6,7,content_c1fe78bc4e37,9,STALE_HIGH_VALUE,REFRESH_NOW
7,8,content_4c76e9b13aea,9,STALE_HIGH_VALUE,REFRESH_NOW
8,9,content_b115f7c74779,9,STALE_HIGH_VALUE,REFRESH_NOW
9,10,content_0919dd345d80,9,STALE_HIGH_VALUE,REFRESH_NOW


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [32]:
top20 = work.head(20).copy()

display(
    top20[
        [
            "rank",
            "content_id",
            "baseline_score",
            "reason_code",
            "action",
            "content_age_days",
            "impressions_90d",
            "ctr",
            "avg_position"
        ]
    ]
)


,rank,content_id,baseline_score,reason_code,action,content_age_days,impressions_90d,ctr,avg_position
0,1,content_36ff89c8214e,9,STALE_HIGH_VALUE,REFRESH_NOW,144,295097,0.05,7.3
1,2,content_c84a0ab98e90,9,STALE_HIGH_VALUE,REFRESH_NOW,95,223271,0.03,7.8
2,3,content_c8e9d6ab9013,9,STALE_HIGH_VALUE,REFRESH_NOW,362,208678,0.00,9.7
3,4,content_91652435f57a,9,STALE_HIGH_VALUE,REFRESH_NOW,257,159590,0.06,7.8
4,5,content_97a86caf3a3d,9,STALE_HIGH_VALUE,REFRESH_NOW,153,147670,0.07,6.4
5,6,content_453722754fea,9,STALE_HIGH_VALUE,REFRESH_NOW,97,140079,0.01,7.6
6,7,content_c1fe78bc4e37,9,STALE_HIGH_VALUE,REFRESH_NOW,153,134055,0.03,7.5
7,8,content_4c76e9b13aea,9,STALE_HIGH_VALUE,REFRESH_NOW,148,127952,0.07,7.4
8,9,content_b115f7c74779,9,STALE_HIGH_VALUE,REFRESH_NOW,313,123469,0.03,8.0
9,10,content_0919dd345d80,9,STALE_HIGH_VALUE,REFRESH_NOW,326,119217,0.02,7.0


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [29]:
# This cell is for CODE (numbers, a query, a check).
# Step 4: Inspect the weakest-looking picks among the top 20

display(
    top20[
        [
            "rank",
            "content_id",
            "baseline_score",
            "reason_code",
            "action",
            "content_age_days",
            "impressions_90d",
            "ctr",
            "avg_position"
        ]
    ].tail(5)
)
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


,rank,content_id,baseline_score,reason_code,action,content_age_days,impressions_90d,ctr,avg_position
15,16,content_d0cc5baa4995,9,STALE_HIGH_VALUE,REFRESH_NOW,148,83651,0.03,6.6
16,17,content_fea6a0d13b4a,9,STALE_HIGH_VALUE,REFRESH_NOW,313,79965,0.07,3.4
17,18,content_a5dbb404bdc2,9,STALE_HIGH_VALUE,REFRESH_NOW,106,79035,0.07,8.7
18,19,content_f0aad930f507,9,STALE_HIGH_VALUE,REFRESH_NOW,419,73457,0.05,5.4
19,20,content_65114d89496d,9,STALE_HIGH_VALUE,REFRESH_NOW,482,72631,0.02,6.5


## Weak picks

Some of the lower-ranked recommendations are weaker because the score
combines several signals without understanding the actual content.

For example, a page may receive staleness points because it is old,
but that does not necessarily mean the content needs to be refreshed.

This is a limitation of the baseline rule and something a future model
could improve.

In [30]:
# Check the columns used by the baseline rule

features_used = [
    "content_age_days",
    "impressions_90d",
    "ctr",
    "avg_position"
]

print("Features used by baseline:")
for feature in features_used:
    print("-", feature)

Features used by baseline:
- content_age_days
- impressions_90d
- ctr
- avg_position


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.